In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import os



# Uncomment the line below to run Chrome in headless mode
# # chrome_options.add_argument("--headless")
driver = webdriver.Chrome()


driver.get("https://parivesh.nic.in/newupgrade/#/trackYourProposal/")

wait = WebDriverWait(driver, 20)

advance_btn = wait.until(
EC.element_to_be_clickable(
    (By.XPATH, "//button[contains(., 'Show Advance Search')]")
    )
    )

advance_btn.click()

In [4]:
# -----------------------------
# Select Major Clearance Type
# -----------------------------
major_clearance = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//select[@formcontrolname='majorClearanceType']")
    )
)

Select(major_clearance).select_by_value("1")


In [5]:
dropdown_element = driver.find_element(By.CSS_SELECTOR, "select[formcontrolname='issueAuthority']")
# 2. Wrap it in Selenium's Select class
select = Select(dropdown_element)

# 3. Select "SEIAA" by its value attribute
select.select_by_value("SEIAA")

In [6]:
# -----------------------------
# Select State
# -----------------------------
state_dropdown = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//select[@formcontrolname='state']")
    )
)
# Fixed: changed state_dropdown_element to state_dropdown to match your variable above
state_select = Select(state_dropdown)

options = [opt.get_attribute("value") for opt in state_select.options if opt.get_attribute("value") != ""]



In [ ]:
# --- Initialize data accumulation storage BEFORE the loop ---
all_table_data = []
headers = []  # We will capture headers on the first successful pass

# -----------------------------
# 1. Get the total number of options first safely
# -----------------------------
state_dropdown = wait.until(
    EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
)
total_options = len(Select(state_dropdown).options)
print(total_options)
# -----------------------------
# 2. Loop by index
# -----------------------------
for index in range(1, total_options):  # Start at 1 to skip placeholder
    #if index == 1:
        #continue
    state_dropdown = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
    )
    state_select = Select(state_dropdown)
    state_value = state_select.options[index].get_attribute("value")
    
    state_select.select_by_index(index)
    print(f"\nProcessing state index {index} (Value: {state_value})...")

    # Click Search Button Safely
    search_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(.,'Search')]"))
    )
    
    existing_tables = driver.find_elements(By.ID, "excel-table")
    old_table = existing_tables[0] if existing_tables else None

    driver.execute_script("arguments[0].click();", search_button)
    print(f"Search successfully forced click for: {state_value}")
    
    if old_table:
        try:
            wait.until(EC.staleness_of(old_table))
        except Exception:
            time.sleep(1) 

    # Handle Missing Table if No Results Exist
    try:
        wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
        print(f"Table successfully loaded for state: {state_value}")
    except TimeoutException:
        print(f"⚠️ No results found (Timeout) for state: {state_value}. Skipping...")
        time.sleep(1)
        continue

    # -----------------------------
    # LIVE ROW CLICKING & SCRAPING LOGIC 
    # -----------------------------
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find("table", {"id": "excel-table"})
    
    if table:
        tbody = table.find("tbody")
        if tbody:
            rows_bs = tbody.find_all("tr")
            
            tbody_text = tbody.get_text(strip=True).lower()
            if "no record" in tbody_text or "no data" in tbody_text or not rows_bs:
                print(f"ℹ️ Table contains explicit empty notice for state: {state_value}. Skipping...")
                time.sleep(1)
                continue

            if not headers:
                thead = table.find("thead")
                if thead:
                    for th in thead.find_all("th"):
                        headers.append(th.get_text(strip=True))

            total_rows = len(rows_bs)
            print(f"Found {total_rows} proposals to process for state {state_value}.")

            # Loop through rows sequentially
            for row_idx in range(1, total_rows + 1):
                try:
                    cols_elements = driver.find_elements(By.XPATH, f"//table[@id='excel-table']/tbody/tr[{row_idx}]/td")
                    row_data = [col.text.strip() for col in cols_elements]
                    
                    if not row_data:
                        continue
                    
                    proposal_link = driver.find_element(By.XPATH, f"//table[@id='excel-table']/tbody/tr[{row_idx}]/td[2]/a")
                    proposal_no = proposal_link.text.strip()
                    
                    print(f" -> Clicking proposal {row_idx}/{total_rows}: {proposal_no}")
                    
                    main_window = driver.current_window_handle
                    driver.execute_script("arguments[0].click();", proposal_link)
                    time.sleep(3)  
                    
                    details_url = "N/A"
                    view_proposal_url = "N/A"
                    
                    opened_in_new_tab = len(driver.window_handles) > 1
                    if opened_in_new_tab:
                        details_window = [w for w in driver.window_handles if w != main_window][0]
                        driver.switch_to.window(details_window)
                    
                    details_url = driver.current_url
                    
                    # -------------------------------------------------------------
                    # NESTED EXTRACTION: Click "View Proposal" & Capture Document URL
                    # -------------------------------------------------------------
                    try:
                        view_proposal_element = driver.find_element(By.XPATH, "//a[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'view proposal')]")
                        pre_click_windows = driver.window_handles
                        
                        driver.execute_script("arguments[0].click();", view_proposal_element)
                        time.sleep(3)
                        
                        post_click_windows = driver.window_handles
                        if len(post_click_windows) > len(pre_click_windows):
                            proposal_doc_window = [w for w in post_click_windows if w not in pre_click_windows][0]
                            driver.switch_to.window(proposal_doc_window)
                            
                            view_proposal_url = driver.current_url  
                            
                            driver.close()  # Close document tab
                            if opened_in_new_tab:
                                driver.switch_to.window(details_window)
                            else:
                                driver.switch_to.window(main_window)
                        else:
                            view_proposal_url = driver.current_url
                            driver.back()
                            time.sleep(1)
                            
                    except NoSuchElementException:
                        print("    ⚠️ 'View Proposal' link/button was not found on this view layout.")
                    except Exception as nested_err:
                        print(f"    ❌ Failed tracking 'View Proposal' sub-route: {nested_err}")
                    
                    # -------------------------------------------------------------
                    # BACKWARDS ROUTING CLEANUP & RE-ALIGNMENT
                    # -------------------------------------------------------------
                    if opened_in_new_tab:
                        driver.close() 
                        driver.switch_to.window(main_window)
                    else:
                        driver.back() 
                        wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
                    
                    # Compile target URL metrics back into data array rows
                    row_data.append(details_url)        
                    row_data.append(view_proposal_url)   
                    row_data.append(state_value)        
                    all_table_data.append(row_data)
                    print(f"    Successfully Scraped URLs.")
                    
                except NoSuchElementException:
                    print(f" ⚠️ Could not find target link anchor for index row {row_idx}. Skipping...")
                    continue
                except Exception as e:
                    print(f" ❌ Error processing index row {row_idx}: {e}")
                    all_windows = driver.window_handles
                    if len(all_windows) > 1:
                        for extra_w in all_windows[1:]:
                            driver.switch_to.window(extra_w)
                            driver.close()
                    driver.switch_to.window(main_window)
                    continue

    time.sleep(1)
    #break  # Kept active for test validation. Remove or comment out to run for all states.

# -----------------------------
# 3. Post-Loop Data Compilation
# -----------------------------
if all_table_data:
    expected_header_count = len(all_table_data[0])
    
    if len(headers) < expected_header_count:
        headers.append("Details_URL")      
        headers.append("Proposal_URL")     
        headers.append("State_Value")
        
    df = pd.DataFrame(all_table_data, columns=headers)
    print("\n--- Final Extracted Dataset Preview ---")
    print(df.head()) 

    file_path = "parivesh_data.csv"

    # Check if the file already exists
    file_exists = os.path.exists(file_path)

    # Write to CSV
    df.to_csv(
    file_path, 
    mode='a', 
    index=False, 
    header=not file_exists  # Writes header ONLY if the file does NOT exist yet
)
    print("\nAll available states scraped and saved to parivesh_data.csv successfully!")
else:
    print("\n❌ Automation complete. Zero data entries found across all processed states.")

In [7]:
# --- Initialize data accumulation storage BEFORE the loop ---
all_table_data = []
headers = []  # We will capture headers on the first successful pass

# -----------------------------
# 1. Get the total number of options first safely
# -----------------------------
state_dropdown = wait.until(
    EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
)
total_options = len(Select(state_dropdown).options)
print(f"Total options found: {total_options}")

# -----------------------------
# 2. Loop by index
# -----------------------------
for index in range(1, total_options):  # Start at 1 to skip placeholder
    state_dropdown = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
    )
    state_select = Select(state_dropdown)
    state_value = state_select.options[index].get_attribute("value")
    
    state_select.select_by_index(index)
    print(f"\nProcessing state index {index} (Value: {state_value})...")

    # Click Search Button Safely
    search_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(.,'Search')]"))
    )
    
    existing_tables = driver.find_elements(By.ID, "excel-table")
    old_table = existing_tables[0] if existing_tables else None

    driver.execute_script("arguments[0].click();", search_button)
    print(f"Search successfully forced click for: {state_value}")
    
    if old_table:
        try:
            wait.until(EC.staleness_of(old_table))
        except Exception:
            time.sleep(1) 

    # Handle Missing Table if No Results Exist
    try:
        wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
        print(f"Table successfully loaded for state: {state_value}")
    except TimeoutException:
        print(f"⚠️ No results found (Timeout) for state: {state_value}. Skipping...")
        time.sleep(1)
        continue

    # -----------------------------
    # LIVE ROW SCRAPING LOGIC 
    # -----------------------------
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find("table", {"id": "excel-table"})
    
    if table:
        tbody = table.find("tbody")
        if tbody:
            rows_bs = tbody.find_all("tr")
            
            tbody_text = tbody.get_text(strip=True).lower()
            if "no record" in tbody_text or "no data" in tbody_text or not rows_bs:
                print(f"ℹ️ Table contains explicit empty notice for state: {state_value}. Skipping...")
                time.sleep(1)
                continue

            if not headers:
                thead = table.find("thead")
                if thead:
                    for th in thead.find_all("th"):
                        headers.append(th.get_text(strip=True))

            total_rows = len(rows_bs)
            print(f"Found {total_rows} proposals to process for state {state_value}.")

            # Loop through table rows sequentially
            for row_idx in range(1, total_rows + 1):
                try:
                    cols_elements = driver.find_elements(By.XPATH, f"//table[@id='excel-table']/tbody/tr[{row_idx}]/td")
                    row_data = [col.text.strip() for col in cols_elements]
                    
                    if not row_data:
                        continue
                    
                    # Append state identity tag to row dataset
                    row_data.append(state_value)        
                    all_table_data.append(row_data)
                    print(f" -> Scraped data row {row_idx}/{total_rows}")
                    
                except Exception as e:
                    print(f" ❌ Error processing index row {row_idx}: {e}")
                    continue

    time.sleep(1)

# -----------------------------
# 3. Post-Loop Data Compilation
# -----------------------------
if all_table_data:
    expected_header_count = len(all_table_data[0])
    
    if len(headers) < expected_header_count:
        headers.append("State_Value")
        
    df = pd.DataFrame(all_table_data, columns=headers)
    print("\n--- Final Extracted Dataset Preview ---")
    print(df.head()) 

    file_path = "parivesh_data.csv"

    # Check if the file already exists to decide header placement
    file_exists = os.path.exists(file_path)

    # Append to CSV cleanly without indices
    df.to_csv(
        file_path, 
        mode='a', 
        index=False, 
        header=not file_exists  # Writes header ONLY if the file does NOT exist yet
    )
    print("\nAll available states scraped and saved to parivesh_data.csv successfully!")
else:
    print("\n❌ Automation complete. Zero data entries found across all processed states.")

Total options found: 37

Processing state index 1 (Value: 35)...
Search successfully forced click for: 35
Table successfully loaded for state: 35
Found 13 proposals to process for state 35.
 -> Scraped data row 1/13
 -> Scraped data row 2/13
 -> Scraped data row 3/13
 -> Scraped data row 4/13
 -> Scraped data row 5/13
 -> Scraped data row 6/13
 -> Scraped data row 7/13
 -> Scraped data row 8/13
 -> Scraped data row 9/13
 -> Scraped data row 10/13
 -> Scraped data row 11/13
 -> Scraped data row 12/13
 -> Scraped data row 13/13

Processing state index 2 (Value: 28)...
Search successfully forced click for: 28
Table successfully loaded for state: 28
Found 4530 proposals to process for state 28.
 -> Scraped data row 1/4530
 -> Scraped data row 2/4530
 -> Scraped data row 3/4530
 -> Scraped data row 4/4530
 -> Scraped data row 5/4530
 -> Scraped data row 6/4530
 -> Scraped data row 7/4530
 -> Scraped data row 8/4530
 -> Scraped data row 9/4530
 -> Scraped data row 10/4530
 -> Scraped data ro

In [8]:
# Quit the WebDriver
driver.quit()

In [ ]:
# imports

import requests
from IPython.display import Markdown, display
import ollama

In [ ]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)
    
pr = Website("https://parivesh.nic.in/")

In [ ]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

# And now: call the Ollama function instead of OpenAI

# Constants

MODEL = "llama3.2"

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']


# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

summarize("https://parivesh.nic.in/")
display_summary("https://anthropic.com")

In [ ]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

display(response.choices[0].message.content)
def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']
chrome_options = webdriver.ChromeOptions()